# Classic DNN Baselines

This notebook runs the final fastText DNN baseline grid across multiple random seeds and creates analysis tables:

- `total_results`: all evaluated DNN combinations across all seeds.
- `per_seed_report_results`: the best validation result per seed and model family.
- `report_results`: mean/std validation metrics across seeds.
- `epoch_history`: per-epoch validation metrics for training curves.

The task predicts sentiment labels `0..4`, with validation score `1 - MAE / 4`.
The DNN workflow intentionally uses fastText only: average fastText document vectors with an MLP plus fastText token sequences with TextCNN and BiLSTM.


In [ ]:
from datetime import datetime
from pathlib import Path
import subprocess
import sys

import pandas as pd

EXPERIMENT_KIND = "FASTTEXT_DNN_BASELINES"
RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
EXPERIMENT_DIR = Path("experiments/classic_dnn") / f"{RUN_ID}_{EXPERIMENT_KIND}"
EMBEDDING_DIR = Path("experiments/embeddings")
EXPERIMENT_DIR.mkdir(parents=True, exist_ok=True)
EMBEDDING_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_PATH = Path("data/train.csv")
FASTTEXT_PATH = EMBEDDING_DIR / "fasttext.vec"
VALIDATION_SIZE = 0.1
SEEDS = [42, 43, 44]
EPOCHS = 10
BATCH_SIZE = 128

FASTTEXT_PATH.exists(), FASTTEXT_PATH, SEEDS


Place the merged fastText vector file at `experiments/embeddings/fasttext.vec`. You can create it with `load_embeddings.ipynb`.


In [ ]:
if not FASTTEXT_PATH.exists():
    raise FileNotFoundError("No fastText vector file found at experiments/embeddings/fasttext.vec. Run load_embeddings.ipynb first.")

completed_runs = []

for seed in SEEDS:
    seed_dir = EXPERIMENT_DIR / f"seed_{seed}"
    seed_dir.mkdir(parents=True, exist_ok=True)
    cmd = [
        sys.executable,
        "-m",
        "baselines.classic_dnn_baselines",
        "--train-path",
        str(TRAIN_PATH),
        "--output-dir",
        str(seed_dir),
        "--validation-size",
        str(VALIDATION_SIZE),
        "--random-state",
        str(seed),
        "--fasttext-path",
        str(FASTTEXT_PATH),
        "--epochs",
        str(EPOCHS),
        "--batch-size",
        str(BATCH_SIZE),
    ]

    print(" ".join(cmd))
    subprocess.run(cmd, check=True)
    completed_runs.append({"seed": seed, "run_dir": seed_dir})

completed_runs


## Total Analysis


In [ ]:
result_frames = []
for run in completed_runs:
    frame = pd.read_csv(run["run_dir"] / "classic_dnn_results.csv")
    frame.insert(0, "seed", run["seed"])
    result_frames.append(frame)

results = pd.concat(result_frames, ignore_index=True)
results.to_csv(EXPERIMENT_DIR / "classic_dnn_results.csv", index=False)

total_results = results.sort_values(
    ["status", "cil_score"], ascending=[False, False]
).reset_index(drop=True)
total_results.to_csv(EXPERIMENT_DIR / "total_analysis.csv", index=False)
total_results


## Report Analysis


In [ ]:
ok_results = results[results["status"] == "ok"].copy()
per_seed_report_results = (
    ok_results.sort_values("cil_score", ascending=False)
    .groupby(["seed", "model"], as_index=False)
    .first()
    .sort_values(["model", "seed"])
    .reset_index(drop=True)
)
per_seed_report_results.to_csv(EXPERIMENT_DIR / "per_seed_report_analysis.csv", index=False)

report_results = (
    per_seed_report_results
    .groupby("model")[["cil_score", "mae", "accuracy", "macro_f1"]]
    .agg(["mean", "std"])
    .reset_index()
)
report_results.columns = [
    column[0] if column[1] == "" else f"{column[0]}_{column[1]}"
    for column in report_results.columns.to_flat_index()
]
report_results = report_results.sort_values("cil_score_mean", ascending=False).reset_index(drop=True)
report_results.to_csv(EXPERIMENT_DIR / "report_analysis.csv", index=False)
report_results


## Epoch History


In [ ]:
history_frames = []
for run in completed_runs:
    frame = pd.read_csv(run["run_dir"] / "classic_dnn_epoch_history.csv")
    frame.insert(0, "seed", run["seed"])
    history_frames.append(frame)

epoch_history = pd.concat(history_frames, ignore_index=True)
epoch_history = epoch_history.sort_values(["experiment", "seed", "epoch"]).reset_index(drop=True)
epoch_history.to_csv(EXPERIMENT_DIR / "classic_dnn_epoch_history.csv", index=False)
epoch_history.to_csv(EXPERIMENT_DIR / "epoch_analysis.csv", index=False)
epoch_history


In [ ]:
best_epochs = (
    epoch_history.sort_values("cil_score", ascending=False)
    .groupby(["seed", "experiment"], as_index=False)
    .first()
    .sort_values("cil_score", ascending=False)
    .reset_index(drop=True)
)
best_epochs.to_csv(EXPERIMENT_DIR / "best_epoch_analysis.csv", index=False)
best_epochs


## Analysis Artifacts


In [ ]:
analysis_dir = EXPERIMENT_DIR / "analysis"
analysis_dir.mkdir(parents=True, exist_ok=True)

dnn_report_table = analysis_dir / "dnn_baseline_table.tex"
dnn_report_csv = analysis_dir / "dnn_baseline_table.csv"
subprocess.run(
    [
        sys.executable,
        "-m",
        "baselines.analysis.make_dnn_report_table",
        "--input",
        str(EXPERIMENT_DIR / "classic_dnn_results.csv"),
        "--output",
        str(dnn_report_table),
        "--summary-csv",
        str(dnn_report_csv),
    ],
    check=True,
)

dnn_report_table, dnn_report_csv
